# **ASSOCIATION RULES**

## **1. Load the cleaned dataset**

In [1]:
import pandas as pd

# Load file
df_clean = pd.read_csv('df_clean.csv')
df_clean.head()

,Unnamed: 0,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail,Revenue,Hour,Day,Month,Year
0,0,1,2023-01-01,1900-01-01 07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg,6.0,7,Sunday,January,2023
1,1,2,2023-01-01,1900-01-01 07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.2,7,Sunday,January,2023
2,2,3,2023-01-01,1900-01-01 07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg,9.0,7,Sunday,January,2023
3,3,4,2023-01-01,1900-01-01 07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm,2.0,7,Sunday,January,2023
4,4,5,2023-01-01,1900-01-01 07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.2,7,Sunday,January,2023


## **2. Running the Apriori Algorithm to Discover Association Rules**

In [6]:
from mlxtend.frequent_patterns import apriori, association_rules

# ==========================================
# 0. CREATE A RECEIPT IDENTIFIER (Receipt_ID)
# Transactions from the same Store + Date + Time are considered part of the same receipt
# ==========================================
df_clean['Receipt_ID'] = (df_clean['store_id'].astype(str) + "_" +
                          df_clean['transaction_date'].astype(str) + "_" +
                          df_clean['transaction_time'].astype(str))

# 1. CREATE MARKET BASKETS
basket = (df_clean.groupby(['Receipt_ID', 'product_detail'])['transaction_qty']
          .sum()
          .unstack()
          .reset_index()
          .fillna(0)
          .set_index('Receipt_ID'))

basket_sets = (basket > 0)

# 2. RUN THE APRIORI ALGORITHM
#  Since the dataset contains over 140,000 records low min_support value (0.001) is used to capture meaningful associations
frequent_itemsets = apriori(basket_sets, min_support=0.001, use_colnames=True)
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# 3. EXPORT RESULTS TO CSV
if rules.empty:
    print("No association rules were found!")
else:
    rules['Primary Product (Purchased))'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
    rules['Associated Product'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

    # Extract key metrics
    rules['Support (%)'] = round(rules['support'] * 100, 3)
    rules['Confidence (%)'] = round(rules['confidence'] * 100, 2)
    rules['Lift'] = round(rules['lift'], 2)

    # Select columns and sort by highest Lift score
    rules_final = rules[['Primary Product (Purchased)', 'Associated Product', 'Support (%)', 'Confidence (%)', 'Lift']]
    rules_final = rules_final.sort_values(by='Lift', ascending=False).reset_index(drop=True)

    # Display the top 15 strongest association rules
    print(rules_final.head(15).to_string())

    # Export to CSV
    rules_final.to_csv('Cross_Selling_Rules.csv', index=False, encoding='utf-8-sig')
    print(f"=> Successfully exported 'Cross_Selling_Rules.csv with {len(rules_final)} association rules!")

  Primary Product (Purchased)       Associated Product  Support (%)  Confidence (%)   Lift
0       Ouro Brasileiro shot              Ginger Scone        0.589           31.29  15.82
1               Ginger Scone      Ouro Brasileiro shot        0.589           29.78  15.82
2                   Latte Rg            Hazelnut syrup        0.318           12.85   9.87
3             Hazelnut syrup                  Latte Rg        0.318           24.39   9.87
4              Cappuccino Lg           Chocolate syrup        0.324           13.68   9.22
5            Chocolate syrup             Cappuccino Lg        0.324           21.81   9.22
6                 Cappuccino  Sugar Free Vanilla syrup        0.339           14.30   9.22
7   Sugar Free Vanilla syrup                Cappuccino        0.339           21.88   9.22
8            Chocolate syrup                  Latte Rg        0.333           22.45   9.08
9                   Latte Rg           Chocolate syrup        0.333           13.47   9.08

# **INSIGHT ANALYSIS AND PRODUCT RECOMMENDATIONS**

**1. Most Frequently Purchased Product Pair**
* **Insight:** The most notable product combination is **Ouro Brasileiro Shot** (coffee) and **Ginger Scone** (pastry). This pair achieved a Lift value of **15.82**, meaning that customers who purchase one item are nearly **16 times more likely** to purchase the other compared to average purchasing behavior. The Confidence score is also relatively high at approximately **30%**.
* **Recommendations:**
  * Create a **"Breakfast Combo"** featuring these two products and offer a small discount to encourage customers to purchase both items together.
  * Place **Ginger Scones** in a highly visible location near the checkout counter or coffee machine so customers can easily add them to their order while waiting for their Ouro Brasileiro coffee.

**2. Customer Preference for Syrup Add-ons in Coffee Drinks**
* **Insight:** Based on association rules #2 through #14, customers show a strong preference for adding flavored syrups to milk-based coffee beverages. These associations have consistently high Lift values ranging from **8.7 to 9.8**.
    * Customers who order **Latte** frequently add **Hazelnut Syrup** or **Chocolate Syrup**.
    * Customers who order **Cappuccino** often add **Sugar-Free Vanilla Syrup**, **Chocolate Syrup**, or **Caramel Syrup**.
* **Recommendations:**
    * Encourage cashiers to proactively suggest syrup add-ons. For example, when a customer orders a Latte, staff could ask: *"Would you like to add Hazelnut or Chocolate Syrup for a richer flavor?"*. Since this behavior is already common among customers, the acceptance rate is likely to be high.
    * Add small recommendation notes below the names of **Latte** and **Cappuccino** on the menu, highlighting the most popular syrup pairings to make customer decisions easier.
